# Localization Analysis — Unitree go2

**Course:** Ciência de Dados — FEI Mestrado  
**Description:** Analysis of RTABMAP localization quality using one or two cameras in a go2 robot.

---

### Data Sources
| File | Description |
|------|-------------|
| `localization_log.csv` | Per-update metrics from `/rtabmap/info` and `/localization_pose` (inliers, covariance, pose, etc.) |
| `plan_log.csv` | Planned path poses from `/plan` topic, grouped by `plan_id` |

### Sections
1. **Data Loading and Data Processing** — read CSV logs from a given run folder and process all data 
2. **Path Visualization** — planned path vs actual robot trajectory  
3. **Localization Quality** — inliers, hypothesis ratio, covariance over time


### 1. **Data Loading and Data Processing**

In [ ]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from scipy.stats.mstats import winsorize
from plotly.subplots import make_subplots




logs_dir = Path("localization_analysis/data/logs")

loc_paths = sorted(logs_dir.glob("logger_csv_*/localization_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
dfs_loc    = [pd.read_csv(loc_path) for loc_path in loc_paths] 

plan_paths = sorted(logs_dir.glob("logger_csv_*/plan_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
dfs_plan    = [pd.read_csv(plan_path) for plan_path in plan_paths] 

[PosixPath('localization_analysis/data/logs/logger_csv_1/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_2/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_3/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_4/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_5/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_6/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_7/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_8/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_9/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_10/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_11/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_12/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/logger_csv_13/plan_log.csv'),
 PosixPath('localization_analysis/data/logs/log

In [2]:
print(f' Localization Dataframes: {len(dfs_loc)}\n Plan Dataframes {len(dfs_plan)}') # Number of DF

 Localization Dataframes: 30
 Plan Dataframes 30


In [3]:
# Take the first path that NAV2 stack calculated (the plan_id = 1)
# Exception: run 26 uses plan_id = 2 (plan_id=1 starts at an outlier position — localization not yet converged)
last_plans = [df[df['plan_id'] == (13 if i == 4 else df['plan_id'].min())] for i, df in enumerate(dfs_plan)]
last_plans[4]

,plan_id,pose_index,x,y
180,13,0,-2.8519,2.1549
181,13,1,-2.8019,2.1049
182,13,2,-2.7519,2.0549
183,13,3,-2.7019,2.0049
184,13,4,-2.6519,1.9549
...,...,...,...,...
328,13,148,1.7117,1.5549
329,13,149,1.7184,1.6049
330,13,150,1.7269,1.6549
331,13,151,1.7370,1.7049


In [4]:
# Drop rows where pose was not yet received (NaN)
actuals = [df.dropna(subset=['pos_x', 'pos_y']) for df in dfs_loc]
actuals

[    timestamp_sec camera_mode  node_id  inliers  matches  inlier_ratio  \
 0    1.775683e+09      double    20082        0        0        0.0000   
 1    1.775683e+09      double    20083        0        0        0.0000   
 2    1.775683e+09      double    20084        0        0        0.0000   
 3    1.775683e+09      double    20085        0        0        0.0000   
 4    1.775683e+09      double    20086        0        0        0.0000   
 5    1.775683e+09      double    20087        0        0        0.0000   
 6    1.775683e+09      double    20088        0        0        0.0000   
 7    1.775683e+09      double    20089        0        0        0.0000   
 8    1.775683e+09      double    20090        0        0        0.0000   
 9    1.775683e+09      double    20091        3       43        0.0079   
 10   1.775683e+09      double    20092        3       29        0.0066   
 11   1.775683e+09      double    20093        3       28        0.0065   
 12   1.775683e+09      d

### 2. **Path Visualization**

In [5]:
n_runs = len(actuals)
cols = 2
rows = math.ceil(n_runs / cols)
titles = [f'Run {i+1}' for i in range(n_runs)]

fig = make_subplots(rows=rows, cols=cols,
                    subplot_titles=titles,
                    shared_yaxes=False)

for i, (actual_df, last_plan) in enumerate(zip(actuals, last_plans)):
    row = i // cols + 1
    col = i % cols + 1

    fig.add_trace(go.Scatter(
        x=last_plan['x'], y=last_plan['y'],
        mode='lines', name='Planned path',
        line=dict(color='royalblue', dash='dash', width=2),
        legendgroup='plan', showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=actual_df['pos_x'], y=actual_df['pos_y'],
        mode='lines', name=f'Run {i+1}',
        line=dict(color='tomato', width=2),
        legendgroup=f'run{i+1}'
    ), row=row, col=col)

fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(
    title='Planned vs Actual Path',
    hovermode='closest',
    height=500 * rows,
    width=900
)
fig.show()


#### 2.1 **Normalizing and get the median of all plans and paths**

##### 2.1.1 **Plans**

In [6]:
def winsorize_and_resample(df, x_col='pos_x', y_col='pos_y', limits=(0.00, 0.00), n_points=500):
    pos_x = np.array(winsorize(df[x_col], limits=limits))
    pos_y = np.array(winsorize(df[y_col], limits=limits))

    coords = np.column_stack([pos_x, pos_y])
    deltas = np.diff(coords, axis=0)
    arc = np.concatenate([[0], np.cumsum(np.hypot(deltas[:, 0], deltas[:, 1]))])
    arc_norm = arc / arc[-1]

    t = np.linspace(0, 1, n_points)
    fx = interp1d(arc_norm, pos_x, kind='linear')
    fy = interp1d(arc_norm, pos_y, kind='linear')
    return fx(t), fy(t)

# Plan paths (use x/y columns)
resampled_plans = [winsorize_and_resample(df, x_col='x', y_col='y') for df in last_plans]

plan_xs = np.array([p[0] for p in resampled_plans])
plan_ys = np.array([p[1] for p in resampled_plans])
median_plan_x = np.median(plan_xs, axis=0)
median_plan_y = np.median(plan_ys, axis=0)


In [7]:
fig = go.Figure()

for i, (px, py) in enumerate(resampled_plans):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Plan {i+1}',
        line=dict(color='tomato', width=1),
        opacity=0.3
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x, y=median_plan_y,
    mode='lines', name='Median plan',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Plans + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()


##### 2.1.1 **Paths**

In [8]:
def winsorize_and_resample(df, limits=(0.00, 0.1), n_points=500):
    # 1. Winsorize para remover outliers
    pos_x = np.array(winsorize(df['pos_x'], limits=limits))
    pos_y = np.array(winsorize(df['pos_y'], limits=limits))

    # 2. Interpolar para n_points uniformes por comprimento de arco
    coords = np.column_stack([pos_x, pos_y])
    deltas = np.diff(coords, axis=0)
    arc = np.concatenate([[0], np.cumsum(np.hypot(deltas[:, 0], deltas[:, 1]))])
    arc_norm = arc / arc[-1]

    t = np.linspace(0, 1, n_points)
    fx = interp1d(arc_norm, pos_x, kind='linear')
    fy = interp1d(arc_norm, pos_y, kind='linear')
    return fx(t), fy(t)

resampled = [winsorize_and_resample(df) for df in actuals]

xs = np.array([p[0] for p in resampled])
ys = np.array([p[1] for p in resampled])

median_x = np.median(xs, axis=0)
median_y = np.median(ys, axis=0)


In [9]:
fig = go.Figure()

for i, (rx, ry) in enumerate(resampled):
    fig.add_trace(go.Scatter(
        x=rx, y=ry,
        mode='lines', name=f'Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=last_plan['x'], y=last_plan['y'],
    mode='lines', name='Planned path',
    line=dict(color='royalblue', dash='dash', width=2)
))
fig.add_trace(go.Scatter(
    x=median_x, y=median_y,
    mode='lines', name='Median path',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Paths + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

### 3. **Localization Quality**